In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import optimizers
from tensorflow.keras.layers import Conv1D, Dense, Flatten, LSTM, MaxPooling1D, TimeDistributed
from tensorflow.keras.models import Sequential
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")

# Fix all RNGs (Python/NumPy/TensorFlow) so this notebook gives the same
# results on every run.
SEED = 42
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

## Helper functions

Shared by every model section below: compute MAE/RMSE/MAPE, and plot actual vs. predicted with a legend.

In [ ]:
def evaluate(name, y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    nonzero = y_true != 0
    mape = (
        np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])) * 100
        if nonzero.any()
        else float("nan")
    )

    print(f"[{name}] MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")
    return {"model": name, "mae": mae, "rmse": rmse, "mape": mape}


def plot_predictions(name, y_true, y_pred):
    plt.figure(figsize=(20, 5))
    plt.plot(np.asarray(y_true).ravel(), color="red", label="Actual")
    plt.plot(np.asarray(y_pred).ravel(), color="blue", label="Predicted")
    plt.title(name)
    plt.legend()
    plt.show()

In [ ]:
# importing the data
raw_csv_data = pd.read_excel("../data/CallCenterData.xlsx")

In [ ]:
# check point of data
df_comp = raw_csv_data.copy()

In [ ]:
df_comp.head()

In [ ]:
df_comp.describe()

In [ ]:
df_comp.isna().sum()

## Setting date as Index

In [ ]:
# taken as a date time field
df_comp.month.describe()

In [ ]:
df_comp.set_index("month", inplace=True)

In [ ]:
df_comp.head(6)

In [ ]:
# seeting the frequency as monthly
df_comp.asfreq('M')

In [ ]:
# seeting the frequency as monthly
df_comp = df_comp.asfreq('M')

In [ ]:
# checking for the null values
df_comp.isna().sum()

## Time Series Visualization

In [ ]:
df_comp.Healthcare.plot(figsize=(20,5), title="Healthcare")
plt.show()

In [ ]:
df_comp.Telecom.plot(figsize=(20,5), title="Telecom")
plt.show()

In [ ]:
df_comp.Banking.plot(figsize=(20,5), title="Banking")
plt.show()

In [ ]:
df_comp.Technology.plot(figsize=(20,5), title="Technology")
plt.show()

In [ ]:
df_comp.Insurance.plot(figsize=(20,5), title="Insurance")
plt.show()

## Setting the training format

In [ ]:
data = df_comp.Healthcare

In [ ]:
#As required for LSTM networks, we require to reshape an input data into n_samples x timesteps x n_features.
#In this example, the n_features is 5. We will make timesteps = 14 (past days data used for training).

#Empty lists to be populated using formatted training data
target_data = []

# Number of days we want to look into the future based on the past days.
n_past = 5  # Number of past days we want to use to predict the future.

#Reformat input data into a shape: (n_samples x timesteps x n_features)
#In my example, my df_for_training_scaled has a shape (?)
#refers to the number of data points and 5 refers to the columns (multi-variables).
for i in range(len(data)):
    temp = []
    for j in range(n_past + 1):
        try:
            temp.append(data[i+j])
        except IndexError:
            continue
    if len(temp) > 5:
        target_data.append(temp)

len(target_data)

## Train / Validation / Test Split

In [ ]:
data_df = pd.DataFrame(target_data, columns=["t-4","t-3","t-2","t-1","t","Y"])
data_df.head()

In [ ]:
# test set split (chronological: held-out, most recent 20 points)
test_size = 20
train = data_df[:-test_size]
test = data_df[-test_size:]

In [ ]:
#LSTM uses sigmoid and tanh that are sensitive to magnitude so values need to be normalized
# normalize the dataset - fit only on the training split, then apply to both splits
scaler = MinMaxScaler(feature_range=(0, 1)) # define min max scaler
scaler = scaler.fit(train)                  # fit the scaler on train data only
train = pd.DataFrame(scaler.transform(train), columns=data_df.columns) # transform the train data
test = pd.DataFrame(scaler.transform(test), columns=data_df.columns)   # transform the test data using the train-fitted scaler

In [ ]:
X, Y = train.drop("Y", axis=1).values, train["Y"].values # drop the column

# validation set split (chronological: most recent slice of what remains after removing test)
valid_fraction = 0.1
n_valid = max(1, round(len(X) * valid_fraction))
X_train, X_valid = X[:-n_valid], X[-n_valid:]
Y_train, Y_valid = Y[:-n_valid], Y[-n_valid:]

X_test, Y_test = test.drop("Y", axis=1).values, test["Y"]

## Neural Networks (Dense) - MLP

In [ ]:
epochs = 100
batch = 256
lr = 0.0003
adam = optimizers.Adam(lr)

In [ ]:
model_mlp = Sequential()
model_mlp.add(Dense(100, activation='relu', input_dim=X_train.shape[1]))
model_mlp.add(Dense(1))
model_mlp.compile(loss='mse', optimizer=adam)
model_mlp.summary()

In [ ]:
mlp_history = model_mlp.fit(X_train, Y_train, validation_data=(X_valid, Y_valid),
                            epochs=epochs, batch_size=batch, verbose=2)

In [ ]:
mlp_pred = model_mlp.predict(X_test)

In [ ]:
plot_predictions("MLP", Y_test, mlp_pred)

In [ ]:
mlp_metrics = evaluate("MLP", Y_test, mlp_pred)

In [ ]:
model_mlp.save("../output/models/mlp_model.keras")

## CNN

In [ ]:
X_train_series = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_valid_series = X_valid.reshape((X_valid.shape[0], X_valid.shape[1], 1))
print('Train set shape', X_train_series.shape)
print('Validation set shape', X_valid_series.shape)

In [ ]:
# a compiled Keras optimizer is bound to one model's variables, so each
# model gets its own fresh Adam instance rather than reusing the MLP's
adam = optimizers.Adam(lr)

model_cnn = Sequential()
model_cnn.add(Conv1D(filters=64, kernel_size=2, activation='relu', input_shape=(X_train_series.shape[1], X_train_series.shape[2])))
model_cnn.add(MaxPooling1D(pool_size=2))
model_cnn.add(Flatten())
model_cnn.add(Dense(50, activation='relu'))
model_cnn.add(Dense(1))
model_cnn.compile(loss='mse', optimizer=adam)
model_cnn.summary()

In [ ]:
cnn_history = model_cnn.fit(X_train_series, Y_train, validation_data=(X_valid_series, Y_valid),
                            epochs=epochs, batch_size=batch, verbose=2)

In [ ]:
X_test_series = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))
cnn_pred = model_cnn.predict(X_test_series)

In [ ]:
plot_predictions("CNN", Y_test, cnn_pred)

In [ ]:
cnn_metrics = evaluate("CNN", Y_test, cnn_pred)

In [ ]:
model_cnn.save("../output/models/cnn_model.keras")

## LSTM

In [ ]:
adam = optimizers.Adam(lr)  # fresh optimizer instance for this model

model_lstm = Sequential()
model_lstm.add(LSTM(50, activation='relu', input_shape=(X_train_series.shape[1], X_train_series.shape[2])))
model_lstm.add(Dense(1))
model_lstm.compile(loss='mse', optimizer=adam)
model_lstm.summary()

In [ ]:
lstm_history = model_lstm.fit(X_train_series, Y_train, validation_data=(X_valid_series, Y_valid),
                              epochs=epochs, batch_size=batch, verbose=2)

In [ ]:
X_test_series = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))
lstm_pred = model_lstm.predict(X_test_series)

In [ ]:
plot_predictions("LSTM", Y_test, lstm_pred)

In [ ]:
lstm_metrics = evaluate("LSTM", Y_test, lstm_pred)

In [ ]:
model_lstm.save("../output/models/lstm_model.keras")

## CNN-LSTM

In [ ]:
X_train_series.shape[1]

In [ ]:
n_past = X_train_series.shape[1]
X_train_series_sub = X_train_series.reshape((X_train_series.shape[0], 1, n_past, 1))
X_valid_series_sub = X_valid_series.reshape((X_valid_series.shape[0], 1, n_past, 1))
print('Train set shape', X_train_series_sub.shape)
print('Validation set shape', X_valid_series_sub.shape)

In [ ]:
adam = optimizers.Adam(lr)  # fresh optimizer instance for this model

model_cnn_lstm = Sequential()
model_cnn_lstm.add(TimeDistributed(Conv1D(filters=64, kernel_size=1, activation='relu'), input_shape=(None, X_train_series_sub.shape[2], X_train_series_sub.shape[3])))
model_cnn_lstm.add(TimeDistributed(MaxPooling1D(pool_size=2)))
model_cnn_lstm.add(TimeDistributed(Flatten()))
model_cnn_lstm.add(LSTM(50, activation='relu'))
model_cnn_lstm.add(Dense(1))
model_cnn_lstm.compile(loss='mse', optimizer=adam)

In [ ]:
cnn_lstm_history = model_cnn_lstm.fit(X_train_series_sub, Y_train, validation_data=(X_valid_series_sub, Y_valid),
                                      epochs=epochs, batch_size=batch, verbose=2)

In [ ]:
n_past_test = X_test_series.shape[1]
X_test_series_sub = X_test_series.reshape((X_test_series.shape[0], 1, n_past_test, 1))
cnn_lstm_pred = model_cnn_lstm.predict(X_test_series_sub)

In [ ]:
plot_predictions("CNN-LSTM", Y_test, cnn_lstm_pred)

In [ ]:
cnn_lstm_metrics = evaluate("CNN-LSTM", Y_test, cnn_lstm_pred)

In [ ]:
model_cnn_lstm.save("../output/models/cnn_lstm_model.keras")

## Model comparison summary

In [ ]:
results = pd.DataFrame([mlp_metrics, cnn_metrics, lstm_metrics, cnn_lstm_metrics]).sort_values("rmse").reset_index(drop=True)
print("Model comparison (sorted by RMSE, on scaled test data):")
results